In [44]:
!pip install -q \
langchain \
langchain-community \
langchain-groq \
chromadb \
sentence-transformers \
unstructured \
beautifulsoup4 \
requests \
langchain-huggingface

**Setting API**

In [45]:
from google.colab import userdata
api_key = userdata.get('New_Groq_apikey')


**Loading document using WebBaseLoader**

In [59]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://docs.langchain.com/"

loader = WebBaseLoader(url)
documents = loader.load()


**Chunking**

In [60]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

**Embeddings**

In [61]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

**Storing Vectors in Chroma DB**

In [62]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [63]:
print(vectorstore._collection.count())

29


**Retrieving the top k chunks using similarity**

In [64]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)


**Defining the LLM model to be used**

In [65]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=api_key
)

**PromptTemplate is structured and sent to llm to augment**

In [66]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY using the context below.
If the answer is not present, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

**Generating the augmented answer**

In [67]:
def rag_answer(question):
    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Combine context
    context = "\n\n".join(doc.page_content for doc in docs)

    # Create prompt
    final_prompt = prompt.format(
        context=context,
        question=question
    )

    # Generate answer
    response = llm.invoke(final_prompt)
    return response.content

In [70]:
question = input("Enter your query: ")
print(rag_answer(question))

Enter your query: what is langchain and key components in langchain?
LangChain is a framework for building conversational AI agents. 

Key components in LangChain include:

1. LangGraph: This is the underlying graph database that provides durable execution, streaming, human-in-the-loop, persistence, and more.
2. Agents: These are built on top of LangGraph and provide a way to create conversational AI agents.
3. Models: These are the machine learning models used by the agents to generate responses.
4. Messages: These are the inputs and outputs of the agents.
5. Tools: These are utilities and functions that can be used to interact with the agents and LangGraph.
6. Short-term memory: This is a component that allows agents to store and retrieve information over short periods of time.
7. Streaming: This is a feature that allows agents to process and respond to input in real-time.
8. Structured output: This is a feature that allows agents to generate structured output, such as tables or list

**Add Conversational Memory**

In [77]:
!pip install -U langchain langchain-community langchain-chroma chromadb python-dotenv sentence-transformers langchain-groq


In [78]:
from dotenv import load_dotenv
import os

from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory


In [79]:
PERSIST_DIR = "db/chroma_db"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embedding_model
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


In [80]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)


In [81]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant that answers questions strictly using the provided context."
    ),

    MessagesPlaceholder("history"),

    (
        "human",
        """Use only the following context to answer the question.
If the answer is not in the context, say you don't have enough information.

Context:
{context}

Question:
{question}
"""
    )
])


In [87]:
rag_chain = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)


In [88]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

rag_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)


In [89]:
def ask(question, session_id="user1"):
    response = rag_with_memory.invoke(
        {"question": question},
        config={"configurable": {"session_id": session_id}}
    )
    print("\nAnswer:\n", response.content)


In [ ]:
print("Ask questions (type 'quit' to exit):")

while True:
    q = input("\nYou: ")
    if q.lower() == "quit":
        break
    ask(q)


Ask questions (type 'quit' to exit):
